In [ ]:
from pathlib import Path
from Physical_Quantity.Physical_Data_Record import SuperCorrRecord
from Physical_Quantity.Super_Correlation_Process import SuperCorrProcessor

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#-----------------########### UltraDMRG计算的三个键方向的超导关联函数<Δ_ij^†·Δ_kl>数据提取与作图 ################----------------------
### 基本参数
N = 200
a = 1.0

### ================= 文件夹路径变量（需要修改的部分）=====================
δ = "d1_100"               # 空穴率
version = "49_50"         # reference bond
bond_dim = "1500"         # bond dimension
t_J = "t_J_3"             # t/J

base_dir = Path(f'Data_N{N}/{t_J}/{δ}/{bond_dim}_measure_super_corr/super_corr_{version}')

### 将三个键方向A、B、C的数据提取并封装成SuperCorrRecord对象列表
records = {}
for bond in 'ABC':
    for channel in 'abcd':
        key = f'scs{bond}{channel}'
        fname = f'{key}_fix_refbond_{version}.json'
        records[key] = SuperCorrRecord.extract_data(
            (base_dir / fname).read_text(encoding='utf-8'), N, a
        )

### 方向显示名映射字典
bond_labels = {
    'A': "A key (blue bond)",
    'B': "B key (green bond)",
    'C': "C key (orange bond)",
}

### 检测超导关联对称性并组合超导关联函数
super_corr_dict_UltraDMRG = {}
for bond, label in bond_labels.items():
    ## 取出当前 bond 对应的四个 SuperCorrRecord 对象
    recs = [records[f'scs{bond}{channel}'] for channel in 'abcd']

    ## 对称性检查
    err_ad, err_bc = SuperCorrProcessor.symmetry_check(*recs)
    print(f"{bond}键方向的通道a与通道d的相对误差为：{err_ad}；通道b与通道c的相对误差为：{err_bc}")

    ## 组合超导关联函数，存入最终字典
    super_corr_dict_UltraDMRG[label] = SuperCorrProcessor.Combined_super_correlation(*recs)

### 可视化
SuperCorrProcessor.Super_Corr_Visualization_three(super_corr_dict_UltraDMRG)
SuperCorrProcessor.Super_Corr_Visualization(super_corr_dict_UltraDMRG[ "A key (blue bond)"], "A key (blue bond)")
SuperCorrProcessor.Super_Corr_Visualization(super_corr_dict_UltraDMRG[ "B key (green bond)"], "B key (green bond)")
SuperCorrProcessor.Super_Corr_Visualization(super_corr_dict_UltraDMRG[ "C key (orange bond)"], "C key (orange bond)")

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#----------########### 超导关联函数衰减类型分析与拟合（SuperCorrDecayAnalyzer） ################---------------------------------------
# 在上一节可视化基础上，对三个键方向分别做衰减类型分析（不修改任何既有数据处理逻辑）
from Physical_Quantity.Super_Correlation_Analyzer import (
    SuperCorrDecayAnalyzer, AnalyzerConfig,
)

### 分析配置（需要调整时修改 AnalyzerConfig 参数）
# envelope_mode 包络模式（重要选项）：
#   'all_points' 全点拟合——窗口内全部 |C(r)| 点做对数空间拟合（样本多，但振荡谷底会拉低包络）
#   'extrema'    峰值法——只用局部极值点拟合包络（不受谷底拉低，但样本少、每峰含噪声）
#   'both'       两种都算：AICc 优者为主包络，另一种作为对比结果写入报告与画图（推荐）
config = AnalyzerConfig(envelope_mode="both")

### 对 A/B/C 三个键方向分别分析
decay_results = {}
for label, rec_list in super_corr_dict_UltraDMRG.items():
    print(f"\n########## 衰减分析键方向：{label} ##########")
    result = SuperCorrDecayAnalyzer.analyze(rec_list, config)
    SuperCorrDecayAnalyzer.print_summary(result)
    decay_results[label] = result

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#----------########### 衰减分析综合可视化（8 面板） ################---------------------------------------------------------------------
import matplotlib.pyplot as plt
from IPython.display import display

### 综合图（默认 standard 模式：2×2 四核心面板）
### 可选 visualization_mode 参数：'standard'（默认 2×2）| 'diagnostic'（1×3 辅助诊断）| 'summary'（纯文本摘要）| 'full'（原 8 面板）
### 关键：display() 显示一次后立刻 plt.close(fig)，防止 Jupyter inline 后端在
### cell 结束时把未关闭的 figure 自动再渲染一次（否则出现两张相同的图）
### 依次输出 A key（索引 0）与 C key（索引 2）两个键方向（title 标注键方向）
for idx in (0, 2):
    label = list(super_corr_dict_UltraDMRG.keys())[idx]
    prep = SuperCorrDecayAnalyzer.group_by_distance(super_corr_dict_UltraDMRG[label])
    fig_main = SuperCorrDecayAnalyzer.Decay_Visualization(decay_results[label], prep.r, prep.C, show=False, visualization_mode = 'standard', title = label)
    display(fig_main)
    plt.close(fig_main)

In [ ]:
#-----------------------------------------------------------------------------------------------------------------------------------------
#----------########### 全部三个键方向批量对比 ################---------------------------------------------------------------------
### 仅并排展示，不自动做跨键方向物理结论
### 关键：display() 显示一次后立刻 plt.close(fig)，避免重复显示
fig_compare = SuperCorrDecayAnalyzer.Decay_Visualization_compare(decay_results, show=False)
display(fig_compare)
plt.close(fig_compare)
